In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DA BASE DE DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: Arquivo {file_path} não encontrado.")
    raise

# ==============================================================================
# 2. BUSCA DA COLUNA DE DATA
# ==============================================================================
cols_data = [c for c in df.columns if 'data' in c.lower() or 'timestamp' in c.lower() or 'date' in c.lower()]

if not cols_data:
    print("ERRO: Nenhuma coluna de data foi encontrada.")
else:
    col_data_alvo = cols_data[0]
    df['Data_Atendimento'] = pd.to_datetime(df[col_data_alvo], errors='coerce')
    df_ev = df.dropna(subset=['Data_Atendimento']).copy()

    if len(df_ev) == 0:
        print("ERRO: As datas encontradas não puderam ser lidas corretamente.")
    else:
        df_ev['Mes'] = df_ev['Data_Atendimento'].dt.month

        # ==============================================================================
        # 3. IDENTIFICAÇÃO DE DOENÇAS RESPIRATÓRIAS (COM A CORREÇÃO DO "UNCHECKED")
        # ==============================================================================
        termos_resp = r'\b(asma|dpoc|bronquite|doença pulmonar obstrutiva)\b'
        text_cols = [c for c in df_ev.columns if df_ev[c].dtype == object]

        cond_texto = df_ev[text_cols].apply(
            lambda col: col.astype(str).str.contains(termos_resp, case=False, na=False, regex=True)
        ).any(axis=1)

        cols_resp_str = [c for c in df_ev.columns if ('asma' in c.lower() or 'dpoc' in c.lower()) and 'choice=' in c.lower() and 'não' not in c.lower()]

        if cols_resp_str:
            cond_str = df_ev[cols_resp_str].apply(
                lambda col: col.astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro'])
            ).any(axis=1)

            df_ev['Caso_Respiratorio'] = (cond_texto | cond_str).astype(int)
        else:
            df_ev['Caso_Respiratorio'] = cond_texto.astype(int)

        # ==============================================================================
        # 4. AGREGAÇÃO MENSAL E DADOS CLIMÁTICOS DE SÃO PAULO
        # ==============================================================================
        df_mensal = df_ev.groupby('Mes').agg(
            Total_Atendimentos=('Record ID', 'count'),
            Casos_Resp=('Caso_Respiratorio', 'sum')
        ).reset_index()

        df_mensal['Taxa_Resp_Perc'] = (df_mensal['Casos_Resp'] / df_mensal['Total_Atendimentos']) * 100

        temp_sp = {
            1: 23.1, 2: 23.5, 3: 22.5, 4: 21.2, 5: 18.4, 6: 17.5,
            7: 16.6, 8: 17.7, 9: 19.3, 10: 20.5, 11: 21.6, 12: 22.4
        }

        nomes_meses = {
            1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun',
            7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'
        }

        meses_completos = pd.DataFrame({'Mes': range(1, 13)})
        df_mensal = meses_completos.merge(df_mensal, on='Mes', how='left').fillna(0)
        df_mensal['Nome_Mes'] = df_mensal['Mes'].map(nomes_meses)
        df_mensal['Temp_Media_SP'] = df_mensal['Mes'].map(temp_sp)

        # ==============================================================================
        # 5. TESTE ESTATÍSTICO (QUI-QUADRADO)
        # ==============================================================================
        casos_observados = df_mensal['Casos_Resp'].values
        total_atend = df_mensal['Total_Atendimentos'].values
        
        # Variáveis padrão para evitar erros na exportação
        chi2_stat, p_val = np.nan, np.nan

        if total_atend.sum() > 0 and casos_observados.sum() > 0:
            taxa_global = casos_observados.sum() / total_atend.sum()
            casos_esperados = total_atend * taxa_global

            mask = total_atend > 0
            if mask.sum() > 1:
                chi2_stat, p_val = chisquare(f_obs=casos_observados[mask], f_exp=casos_esperados[mask])
                if p_val < 0.05:
                    significancia = f"Estatisticamente Significativo (p = {p_val:.4f})"
                    cor_titulo = '#c0392b'
                else:
                    significancia = f"Não Significativo (p = {p_val:.4f})"
                    cor_titulo = '#7f8c8d'
            else:
                significancia = "Dados insuficientes para teste"
                cor_titulo = '#000000'
        else:
            significancia = "Sem dados de casos respiratórios"
            cor_titulo = '#000000'

        # ==============================================================================
        # 6. EXPORTAÇÃO DAS BASES DE DADOS (CSVs PARA O ARTIGO)
        # ==============================================================================
        # Base 1: Tabela Sazonal Mensal (Eixo X do Gráfico)
        tabela_export = df_mensal[['Mes', 'Nome_Mes', 'Total_Atendimentos', 'Casos_Resp', 'Taxa_Resp_Perc', 'Temp_Media_SP']].copy()
        tabela_export.columns = ['Mês (Numérico)', 'Mês', 'Total de Atendimentos (N)', 'Casos Respiratórios (N)', 'Taxa de Casos (%)', 'Temperatura Média SP (°C)']
        
        # Arredonda a taxa para ficar bonito no Excel
        tabela_export['Taxa de Casos (%)'] = tabela_export['Taxa de Casos (%)'].round(2)
        tabela_export.to_csv('base_sazonalidade_respiratoria.csv', index=False, encoding='utf-8-sig')

        # Base 2: Tabela de Resultados Estatísticos
        tabela_stats = pd.DataFrame({
            'Métrica': [
                'Total de Atendimentos Analisados', 
                'Total de Casos Respiratórios (Asma/DPOC/Bronquite)', 
                'Estatística Qui-Quadrado (X²)', 
                'Valor-p (Significância)', 
                'Interpretação Sazonal'
            ],
            'Valor': [
                total_atend.sum(), 
                casos_observados.sum(), 
                round(chi2_stat, 4) if not np.isnan(chi2_stat) else "N/A", 
                f"{p_val:.4e}" if not np.isnan(p_val) else "N/A", 
                significancia.split(" (")[0]
            ]
        })
        tabela_stats.to_csv('base_estatistica_sazonalidade.csv', index=False, encoding='utf-8-sig')

        print(f"\n{'='*80}")
        print("✅ BASES DE DADOS EXPORTADAS COM SUCESSO:")
        print("- 'base_sazonalidade_respiratoria.csv' (Mês a Mês: N, Taxas e Temperatura)")
        print("- 'base_estatistica_sazonalidade.csv' (Resultados do Teste Qui-Quadrado)")
        print(f"{'='*80}")

        # ==============================================================================
        # 7. VISUALIZAÇÃO CLIMÁTICA-CLÍNICA
        # ==============================================================================
        sns.set_theme(style="whitegrid")
        fig, ax1 = plt.subplots(figsize=(14, 8))

        cor_barras = '#3498db'
        sns.barplot(data=df_mensal, x='Nome_Mes', y='Taxa_Resp_Perc', color=cor_barras, alpha=0.8, ax=ax1)
        ax1.set_ylabel('% de Casos Respiratórios (Asma/DPOC) no Mês', color=cor_barras, fontsize=13, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=cor_barras)
        ax1.set_xlabel('Mês do Ano', fontsize=13, fontweight='bold')
        ax1.grid(False)

        ax2 = ax1.twinx()
        cor_linha = '#e74c3c'
        sns.lineplot(data=df_mensal, x='Nome_Mes', y='Temp_Media_SP', color=cor_linha, marker='o',
                     linewidth=3, markersize=10, ax=ax2)
        ax2.set_ylabel('Temperatura Média de São Paulo (°C)', color=cor_linha, fontsize=13, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor=cor_linha)

        ax2.set_ylim(bottom=0, top=30)

        plt.title(f"Sazonalidade: Doenças Respiratórias vs. Temperatura (SP)\nVariância Sazonal: {significancia}",
                  fontsize=16, fontweight='bold', pad=20, color=cor_titulo)

        for i, p in enumerate(ax1.patches):
            if p.get_height() > 0:
                ax1.annotate(f"{p.get_height():.1f}%",
                             (p.get_x() + p.get_width() / 2., p.get_height()),
                             ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', fontweight='bold', color='#2c3e50')

        fig.tight_layout()
        plt.show()

        # ==============================================================================
        # 8. RELATÓRIO DE IMPACTO SAZONAL NO CONSOLE
        # ==============================================================================
        print(f"\n{'='*80}")
        print("SÍNTESE EPIDEMIOLÓGICA DE SAZONALIDADE")
        print(f"{'='*80}")
        print(f" -> Atendimentos analisados (com data): {total_atend.sum()}")
        print(f" -> Total REAL de episódios de Asma/DPOC/Bronquite: {casos_observados.sum()}")
        print(f" -> Diagnóstico Estatístico: {significancia}")

        if not np.isnan(p_val) and p_val < 0.05:
            mes_pico = df_mensal.loc[df_mensal['Taxa_Resp_Perc'].idxmax()]
            print(f"\n[ALERTA DE GESTÃO]")
            print(f" -> A variação é real. O pico ocorreu em {mes_pico['Nome_Mes']} ({mes_pico['Taxa_Resp_Perc']:.1f}% da demanda total).")
            print(f" -> A temperatura média de {mes_pico['Nome_Mes']} na cidade de SP foi de {mes_pico['Temp_Media_SP']}°C.")
        print(f"{'='*80}\n")